# Data Preparation

In [ ]:
import numpy as np
import pandas as pd
import torch
import re

In [ ]:
!nvidia-smi 

'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
import torch
if torch.cuda.is_available():
  print("CUDA available. GPU will be used for computation.")
  device = 0 # Default to the first GPU; adjust if you have multiple GPUs
else:
  print("CUDA not available. CPU will be used for computation.")
  device = -1 # Indicates CPU usage

CUDA not available. CPU will be used for computation.


In [ ]:
# 1. Load Data
df = pd.read_csv("News.csv")
df.head()

,id,id_author,title,portal,time,author,editor,content,source
0,0,1,Infografis Pekerja Asing Dilarang Masuk Wilaya...,Liputan6.com,"24 Jul 2021, 09:02 WIB",Abdillah,Abdillah,Pemerintah melalui Menteri Hukum dan Hak Asasi...,https://www.liputan6.com/news/read/4614451/inf...
1,1,1,Infografis Jadwal Bulu Tangkis Indonesia di Ol...,Liputan6.com,"23 Jul 2021, 23:23 WIB",Abdillah,Abdillah,Bulu Tangkis menjadi andalan Indonesia berburu...,https://www.liputan6.com/bola/read/4614427/inf...
2,2,1,"Infografis Jangan Bebal, Kamu Tidak Kebal Covi...",Liputan6.com,"23 Jul 2021, 10:40 WIB",Abdillah,Abdillah,Covid-19 tidak mengenal usia dan status. Siapa...,https://www.liputan6.com/news/read/4613233/inf...
3,3,1,Infografis Awas Perokok Lebih Rentan Tertular ...,Liputan6.com,"22 Jul 2021, 10:35 WIB",Abdillah,Abdillah,Kebiasaan merokok berisiko menimbulkan sejumla...,https://www.liputan6.com/news/read/4612324/inf...
4,4,1,Infografis Perbedaan Aturan PPKM Level 3 dan 4,Liputan6.com,"22 Jul 2021, 09:01 WIB",Abdillah,Abdillah,Pemberlakuan Pembatasan Kegiatan Masyarakat at...,https://www.liputan6.com/news/read/4612511/inf...


In [ ]:
# 2. Clean data
df = df.dropna(subset=["content"])  # drop rows where 'content' is missing

def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)   # remove URLs
    text = re.sub(r"<.*?>", " ", text)               # remove HTML tags
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)       # remove punctuation/special chars
    text = re.sub(r"\s+", " ", text).strip()          # collapse multiple spaces
    return text

df["cleaned_content"] = df["content"].apply(clean_text)
df.head(3)

,id,id_author,title,portal,time,author,editor,content,source,cleaned_content
0,0,1,Infografis Pekerja Asing Dilarang Masuk Wilaya...,Liputan6.com,"24 Jul 2021, 09:02 WIB",Abdillah,Abdillah,Pemerintah melalui Menteri Hukum dan Hak Asasi...,https://www.liputan6.com/news/read/4614451/inf...,Pemerintah melalui Menteri Hukum dan Hak Asasi...
1,1,1,Infografis Jadwal Bulu Tangkis Indonesia di Ol...,Liputan6.com,"23 Jul 2021, 23:23 WIB",Abdillah,Abdillah,Bulu Tangkis menjadi andalan Indonesia berburu...,https://www.liputan6.com/bola/read/4614427/inf...,Bulu Tangkis menjadi andalan Indonesia berburu...
2,2,1,"Infografis Jangan Bebal, Kamu Tidak Kebal Covi...",Liputan6.com,"23 Jul 2021, 10:40 WIB",Abdillah,Abdillah,Covid-19 tidak mengenal usia dan status. Siapa...,https://www.liputan6.com/news/read/4613233/inf...,Covid 19 tidak mengenal usia dan status Siapa ...


In [ ]:
# 3 & 4. Tokenization + Lowercasing
def tokenize(text):
    text = text.lower()          # lowercasing
    tokens = text.split()        # simple whitespace tokenization
    return tokens

df["tokens"] = df["cleaned_content"].apply(tokenize)
df[["content", "tokens"]].head(3)

,content,tokens
0,Pemerintah melalui Menteri Hukum dan Hak Asasi...,"[pemerintah, melalui, menteri, hukum, dan, hak..."
1,Bulu Tangkis menjadi andalan Indonesia berburu...,"[bulu, tangkis, menjadi, andalan, indonesia, b..."
2,Covid-19 tidak mengenal usia dan status. Siapa...,"[covid, 19, tidak, mengenal, usia, dan, status..."


In [ ]:
# 5. Build Vocabulary of Unique Terms
from collections import Counter

# Flatten all tokens across all documents into one big list
all_tokens = [token for tokens in df["tokens"] for token in tokens]

# Count frequency of each term across the corpus (useful for inspection, not just vocab)
term_freq = Counter(all_tokens)

# Build vocabulary: sorted list of unique terms
vocab = sorted(term_freq.keys())

# Map each term to an index (needed later for tf/tf-idf vector construction)
vocab_to_idx = {term: idx for idx, term in enumerate(vocab)}

print(f"Total tokens in corpus: {len(all_tokens)}")
print(f"Vocabulary size (unique terms): {len(vocab)}")
print("Sample vocab terms:", vocab[:10])

Total tokens in corpus: 5690989
Vocabulary size (unique terms): 90462
Sample vocab terms: ['0', '00', '000', '0000', '0001', '00010', '000100', '0001000', '0002', '00023']


In [ ]:
# Most common terms
most_common_df = pd.DataFrame(term_freq.most_common(15), columns=["term", "frequency"])

# Least common terms
least_common_df = pd.DataFrame(list(term_freq.items())[-15:], columns=["term", "frequency"])

print("Most common terms:")
display(most_common_df)

print("Least common terms (sample):")
display(least_common_df)

Most common terms:


,term,frequency
0,yang,131750
1,dan,115197
2,di,101584
3,untuk,57911
4,ini,57851
5,dengan,52194
6,dari,48863
7,itu,41400
8,dalam,39489
9,pada,32856


Least common terms (sample):


,term,frequency
0,komcad,23
1,psdn,3
2,kesukarelaan,1
3,bsnp,1
4,mengonversinya,1
5,hirarki,1
6,militeristik,3
7,tjetjep,4
8,kesarjanaan,2
9,berpatok,1


In [ ]:
# Create 5 queries
# Check docs include candidate topics
candidate_keywords = ["covid", "vaksin", "pajak", "pendidikan", "konsumen", "ekonomi", "pemerintah", "corona"]

for kw in candidate_keywords:
    count = df["cleaned_content"].str.contains(kw, case=False, na=False).sum()
    print(f"'{kw}': {count} dokumen")

'covid': 6469 dokumen
'vaksin': 2318 dokumen
'pajak': 323 dokumen
'pendidikan': 720 dokumen
'konsumen': 336 dokumen
'ekonomi': 1913 dokumen
'pemerintah': 4972 dokumen
'corona': 1457 dokumen


In [ ]:
# Final 5 queries
queries = [
    "kasus covid 19",
    "vaksinasi covid",
    "kebijakan pemerintah",
    "dampak ekonomi",
    "kasus pajak"
]

# Build Retrieval Models

### TF Representation

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import time

# Reconstruct cleaned/tokenized text as space-joined strings (sklearn expects strings, not token lists)
df["joined_tokens"] = df["tokens"].apply(lambda tokens: " ".join(tokens))

# Build the tf matrix using the vocab we already created
vectorizer_tf = CountVectorizer(vocabulary=vocab)
tf_matrix = vectorizer_tf.fit_transform(df["joined_tokens"])

print("TF matrix shape (documents x vocab terms):", tf_matrix.shape)

TF matrix shape (documents x vocab terms): (14334, 90462)


In [ ]:
def preprocess_query(query):
    query = query.lower()
    query = re.sub(r"[^a-zA-Z0-9\s]", " ", query)
    return " ".join(query.split())

def tf_search(query, top_k=10):
    start = time.time()
    
    query_clean = preprocess_query(query)
    query_vec = vectorizer_tf.transform([query_clean])
    
    scores = cosine_similarity(query_vec, tf_matrix).flatten()
    top_indices = scores.argsort()[::-1][:top_k]
    
    elapsed = time.time() - start
    
    results = df.iloc[top_indices][["content"]].copy()
    results["score"] = scores[top_indices]
    
    return results, elapsed

# Example usage
results, elapsed = tf_search("economy stock market")
print(f"TF search time: {elapsed:.4f} seconds")
print(results.head(10))

TF search time: 0.0626 seconds
                                                content     score
8750    Bank Central Asia atau BCA memutuskan untuk ...  0.138675
7625    Wakil Presiden (Wapres), Ma'ruf Amin mengata...  0.093082
8751    Harga saham PT Bank Central Asia Tbk (BBCA) ...  0.090691
6270    Global Islamic Report 2020 mencatat perkemba...  0.080582
3570  PT Garudafood Putra Putri Jaya Tbk (GOOD) meng...  0.077870
864    Pada masa pandemi COVID-19, digitalisasi menj...  0.074019
668     Menteri Badan Usaha Milik Negara (BUMN) Eric...  0.064730
7626    Wakil Presiden (Wapres), Ma'ruf Amin menyebu...  0.051743
4030   Memasak menjadi hobi yang sangat diganrungi s...  0.049281
3512  PT Mandiri Sekuritas (Mansek) mencatat kenaika...  0.047378


In [ ]:
# Run all 5 query in model TF
tf_results = {}
tf_times = {}

for q in queries:
    results, elapsed = tf_search(q, top_k=10)
    tf_results[q] = results
    tf_times[q] = elapsed
    print(f"\n=== Query: '{q}' | waktu: {elapsed:.4f} detik ===")
    display(results[["content", "score"]])


=== Query: 'kasus covid 19' | waktu: 0.0711 detik ===


,content,score
13861,Kasus terkonfirmasi positif Covid-19 di Indo...,0.883596
13796,Kasus terkonfirmasi positif Covid-19 di Indo...,0.883022
13848,Kasus terkonfirmasi positif Covid-19 di Indo...,0.883001
13815,Kasus terkonfirmasi positif Covid-19 di Indo...,0.882281
13825,Kasus terkonfirmasi positif Covid-19 di Indo...,0.881119
1090,PT Wijaya Karya (Persero) Tbk. (WIKA) menyerah...,0.868790
2454,Pemerintah melaporkan terdapat 34.379 kasus ba...,0.783841
11940,Kasus positif Covid-19 di tanah air kembali ...,0.783832
2357,Pemerintah melaporkan terdapat 49.071 kasus ba...,0.777444
2459,Pemerintah melaporkan terdapat 31.189 kasus b...,0.777140



=== Query: 'vaksinasi covid' | waktu: 0.0522 detik ===


,content,score
8108,Warga DKI Jakarta bisa melakukan pendaftaran...,0.531975
2891,Ribuan pelajar dari berbagai SMP dan SMA di DK...,0.482742
2423,Kepala Staf Kepresidenan Moeldoko mengatakan v...,0.480166
4817,UGM tengah mendata para pegawainya untuk diaju...,0.475164
7726,"Wakil Presiden Ma'ruf Amin mengatakan, vaksi...",0.463988
1519,Markas Komando Daerah Militer (Makodam) V/Braw...,0.460287
1415,Program vaksinasi Gotong Royong (VGR) Individu...,0.459901
1904,Pemerintah Daerah (Pemda) Provinsi Jawa Barat...,0.456738
14062,Kementerian Kesehatan mengizinkan penggunaan...,0.456351
12607,Relawan Arus Bawah Jokowi (ABJ) mendukung se...,0.454771



=== Query: 'kebijakan pemerintah' | waktu: 0.0465 detik ===


,content,score
5997,Badan Kepegawaian Negara (BKN) menyampaikan ...,0.457829
7631,Badan Kepegawaian Negara (BKN) melaporkan ju...,0.453692
7642,Badan Kepegawaian Negara (BKN) mencatat juml...,0.452988
7616,Badan Kepegawaian Negara (BKN) mencatat juml...,0.442963
7609,Badan Kepegawaian Negara (BKN) mencatat juml...,0.433577
6015,Badan Kepegawaian Negara (BKN) mencatat seba...,0.429605
7581,Badan Kepegawaian Negara (BKN) mencatat juml...,0.420622
6221,Ketua Asosiasi Pengelola Pusat Belanja Indon...,0.418237
7579,Badan Kepegawaian Negara (BKN) mencatat juml...,0.412826
7568,Badan Kepegawaian Negara (BKN) mencatat juml...,0.405826



=== Query: 'dampak ekonomi' | waktu: 0.0528 detik ===


,content,score
6074,Menteri Keuangan Sri Mulyani Indrawati menga...,0.340373
6141,Menteri Kesehatan Budi Gunadi Sadikin menila...,0.314786
6075,Wakil Presiden Ma'ruf Amin menghadiri launch...,0.281893
11851,"Presiden Joko Widodo atau Jokowi mengatakan,...",0.272798
5974,Presiden Joko Widodo (Jokowi) memperkirakan ...,0.264039
1071,Pertumbuhan ekonomi Indonesia diproyeksi melam...,0.264039
2703,Bank Indonesia (BI) memprediksi pertumbuhan ek...,0.256424
6111,Karantina wilayah secara total atau lockdown...,0.253917
5985,"Menteri Pariwisata dan Ekonomi Kreatif, Sand...",0.250000
6206,"Deputi Gubernur Bank Indonesia (BI), Doni Pr...",0.242393



=== Query: 'kasus pajak' | waktu: 0.0437 detik ===


,content,score
13995,Penambahan kasus positif Covid-19 di Indones...,0.579480
14033,Kasus positif Covid-19 di Indonesia mengalam...,0.579480
14024,Kasus positif Covid-19 di Indonesia mengalam...,0.578096
13944,Kasus terkonfirmasi positif Covid-19 mengala...,0.575924
13881,Kementerian Kesehatan kembali merilis hasil ...,0.568216
13932,Kasus terkonfirmasi positif Covid-19 di Indo...,0.563771
13215,Penularan Covid-19 semakin masif di Kalimant...,0.562772
4447,Pengamat Pajak Danny Darussalam Tax Center (DD...,0.547120
13923,Kasus terkonfirmasi positif Covid-19 di Indo...,0.543086
13913,Kasus terkonfirmasi positif Covid-19 di Indo...,0.531162


### TF-IDF Representation

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Build the tf-idf matrix using the same vocab we already established
vectorizer_tfidf = TfidfVectorizer(vocabulary=vocab)
tfidf_matrix = vectorizer_tfidf.fit_transform(df["joined_tokens"])

print("TF-IDF matrix shape (documents x vocab terms):", tfidf_matrix.shape)

TF-IDF matrix shape (documents x vocab terms): (14334, 90462)


In [ ]:
def tfidf_search(query, top_k=10):
    start = time.time()
    
    query_clean = preprocess_query(query)
    query_vec = vectorizer_tfidf.transform([query_clean])
    
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_indices = scores.argsort()[::-1][:top_k]
    
    elapsed = time.time() - start
    
    results = df.iloc[top_indices][["content"]].copy()
    results["score"] = scores[top_indices]
    
    return results, elapsed

In [ ]:
tfidf_results = {}
tfidf_times = {}

for q in queries:
    results, elapsed = tfidf_search(q, top_k=10)
    tfidf_results[q] = results
    tfidf_times[q] = elapsed
    print(f"\n=== Query: '{q}' | waktu: {elapsed:.4f} detik ===")
    display(results[["content", "score"]])


=== Query: 'kasus covid 19' | waktu: 0.0473 detik ===


,content,score
13913,Kasus terkonfirmasi positif Covid-19 di Indo...,0.685914
13861,Kasus terkonfirmasi positif Covid-19 di Indo...,0.673825
13848,Kasus terkonfirmasi positif Covid-19 di Indo...,0.672682
13923,Kasus terkonfirmasi positif Covid-19 di Indo...,0.671805
13796,Kasus terkonfirmasi positif Covid-19 di Indo...,0.665168
13815,Kasus terkonfirmasi positif Covid-19 di Indo...,0.664157
13825,Kasus terkonfirmasi positif Covid-19 di Indo...,0.662891
13932,Kasus terkonfirmasi positif Covid-19 di Indo...,0.657214
1090,PT Wijaya Karya (Persero) Tbk. (WIKA) menyerah...,0.652731
13944,Kasus terkonfirmasi positif Covid-19 mengala...,0.634597



=== Query: 'vaksinasi covid' | waktu: 0.0499 detik ===


,content,score
7726,"Wakil Presiden Ma'ruf Amin mengatakan, vaksi...",0.492547
8030,Juru Bicara Vaksinasi Covid-19 Kementerian K...,0.469398
2423,Kepala Staf Kepresidenan Moeldoko mengatakan v...,0.467713
14062,Kementerian Kesehatan mengizinkan penggunaan...,0.461905
8108,Warga DKI Jakarta bisa melakukan pendaftaran...,0.455594
1415,Program vaksinasi Gotong Royong (VGR) Individu...,0.440642
8060,"Pemerintah telah berhasil mencapai target 1,...",0.426202
1900,"Rencana vaksinasi Gotong Royong individu, ata...",0.420988
9405,Rumah Sakit Umum Pusat (RSUP) M Djamil Padan...,0.416466
12820,Kementerian Kesehatan (Kemenkes) optimis Ind...,0.415936



=== Query: 'kebijakan pemerintah' | waktu: 0.0452 detik ===


,content,score
14147,"Sosiolog dari Universitas Gadjah Mada (UGM),...",0.382883
13930,Pandemi Covid-19 masih melanda banyak negara...,0.358856
10897,Pemerintah akan menerapkan kebijakan Pemberl...,0.349760
6221,Ketua Asosiasi Pengelola Pusat Belanja Indon...,0.339103
6348,Presiden Joko Widodo (Jokowi) menetapkan Pem...,0.328293
11960,Anggota Komisi IX DPR Fraksi PAN Saleh Daula...,0.328163
5315,Ketua Fraksi Partai Amanat Nasional (PAN) DP...,0.325780
5179,Pemerintah belum juga memutuskan kebijakan P...,0.318083
12750,"Anggota Komisi VI DPR RI, Deddy Sitorus meng...",0.315336
2702,Pemerintah mengganti istilah PPKM Darurat menj...,0.303446



=== Query: 'dampak ekonomi' | waktu: 0.0710 detik ===


,content,score
6074,Menteri Keuangan Sri Mulyani Indrawati menga...,0.297221
6141,Menteri Kesehatan Budi Gunadi Sadikin menila...,0.292180
5974,Presiden Joko Widodo (Jokowi) memperkirakan ...,0.245341
11851,"Presiden Joko Widodo atau Jokowi mengatakan,...",0.238470
6075,Wakil Presiden Ma'ruf Amin menghadiri launch...,0.230157
7600,Pemerintah tengah melakukan evaluasi terkait...,0.225548
5985,"Menteri Pariwisata dan Ekonomi Kreatif, Sand...",0.223948
2703,Bank Indonesia (BI) memprediksi pertumbuhan ek...,0.219819
7570,Kamar Dagang dan Industri (Kadin) Indonesia ...,0.211466
1071,Pertumbuhan ekonomi Indonesia diproyeksi melam...,0.201356



=== Query: 'kasus pajak' | waktu: 0.0493 detik ===


,content,score
4447,Pengamat Pajak Danny Darussalam Tax Center (DD...,0.761227
4449,"Bertepatan dengan momentum Hari Pajak 2021, Di...",0.703788
11288,"Sekretaris Daerah (Sekda) Provinsi Bali, Dew...",0.628627
6228,Pemerintah Jokowi memperluas pemajakan melal...,0.538694
1817,Tim penyidik Komisi Pemberatasan Korupsi (KPK)...,0.450886
1634,"Selama bertahun-tahun, laporan pajak dari kepe...",0.441464
6117,"Pemerintah Kabupaten (Pemkab) Karawang, Jawa...",0.430635
6085,Penelitian Big Data Continuum Indonesia mela...,0.426520
14256,"Wakil Ketua Fraksi NasDem DPR RI, Willy Adit...",0.425077
10123,"Wakil Ketua MPR RI Arsul Sani, ikut berpenda...",0.420917


### Word2Vec (Pretrained)

In [ ]:
%pip install gensim

  Using cached gensim-4.4.0.tar.gz (23.3 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached smart_open-8.0.1-py3-none-any.whl.metadata (24 kB)
  Using cached wrapt-2.4.1-cp314-cp314-win_amd64.whl.metadata (7.6 kB)
Using cached smart_open-8.0.1-py3-none-any.whl (73 kB)
Using cached wrapt-2.4.1-cp314-cp314-win_amd64.whl (98 kB)
Failed to build gensim
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Building wheel for gensim (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [716 lines of output]
      C:\Users\Rizka Akmalia\AppData\Local\Temp\pip-build-env-zml3x090\overlay\Lib\site-packages\setuptools\_distutils\dist.py:318: UserWarning: Unknown distribution option: 'test_suite'
        warnings.warn(msg)
      C:\Users\Rizka Akmalia\AppData\Local\Temp\pip-build-env-zml3x090\overlay\Lib\site-packages\setuptools\_distutils\dist.py:318: UserWarning: Unknown distribution option: 'tests_require'
        warnings.warn(msg)
      running bdist_wheel
      running build
      running build_py
      creating build\lib.win-amd64-cpython-314\gensim
      copying gensim\downloader.py -> build\lib.win-amd64-cpython-314\gensim
      copying gensim\interfaces.py -> build\lib.win-amd64-cpython-314\gensim
      copying gensim\matutils.py -> build\lib.win-amd64-cpython-314\gensim
      copying gensim\nosy.py -> build\lib.win-amd64-cpytho

In [ ]:
import gensim.downloader as api
from gensim.models import KeyedVectors
import urllib.request
import gzip
import shutil
import os

# URL resmi FastText pretrained vectors untuk bahasa Indonesia
url = "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.id.300.vec.gz"
gz_path = "cc.id.300.vec.gz"
vec_path = "cc.id.300.vec"

# Download (hanya sekali, cukup besar ~1.3GB, jadi mungkin butuh waktu)
if not os.path.exists(vec_path):
    print("Downloading pretrained FastText Indonesian vectors...")
    urllib.request.urlretrieve(url, gz_path)
    
    print("Extracting...")
    with gzip.open(gz_path, "rb") as f_in:
        with open(vec_path, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)

# Load hanya sebagian vocab teratas untuk hemat memori (misal 200,000 kata paling umum)
print("Loading model...")
word2vec_model = KeyedVectors.load_word2vec_format(vec_path, limit=200000)

print("Model loaded. Vector size:", word2vec_model.vector_size)

ModuleNotFoundError: No module named 'gensim'